In [1]:
#%%
import time
import numpy as np
import pandas as pd

from pb_differential_flux_bis import differential_surface_flux

In [6]:

def build_differential_table(
    mass_values_gev,
    eps_values,
    cos_theta_values,
    Echi_grid,
    **kwargs
):
    rows = []

    for mchi in mass_values_gev:
        for eps in eps_values:
            flux = differential_surface_flux(
                mchi=mchi,
                Echi=Echi_grid,
                eps=eps,
                **kwargs
            )

            for energy, dphi in zip(Echi_grid, flux):
                for cos_theta in cos_theta_values:
                    rows.append({
                        "eps2": float(eps**2),
                        "cos_theta": float(cos_theta),
                        "m(GeV)": float(mchi),
                        "Energy": float(energy),
                        "flux_differential": float(dphi),
                    })

    return pd.DataFrame(rows)


t0 = time.time()
# --- ejemplo de uso ---
mass_values_gev = np.geomspace(0.01,1.5, 76)         # 100 MeV
eps_values = [1.0]           # epsilon = 1e-1  => eps^2 = 1e-2
cos_theta_values = [1.0]
Echi_grid = np.geomspace(1e-1, 30.0, 50)

df = build_differential_table(
    mass_values_gev=mass_values_gev,
    eps_values=eps_values,
    cos_theta_values=cos_theta_values,
    Echi_grid=Echi_grid,
    Ep_max=1e4,
    n_Ep=30,
    n_k2=100,
    n_E0=60,
    n_cos=80
)

# Seguridad: elimina eps si viene de ejecuciones anteriores
df = df.drop(columns=["eps"], errors="ignore")

print(df.head())
df.to_csv("PB_differential_flux_table.csv", index=False)
print("Archivo guardado: PB_differential_flux_table.csv")
print(f"Tiempo de ejecución: {time.time() - t0:.3f} s")

KeyboardInterrupt: 

In [7]:
# Validación rápida de columnas (corrida corta)
mass_values_test = np.geomspace(0.05, 0.08, 2)
eps_values_test = [1.0]
cos_theta_values_test = [0.0, 1.0]
Echi_grid_test = np.geomspace(1e-1, 1.0, 6)

t_test = time.time()
df_test = build_differential_table(
    mass_values_gev=mass_values_test,
    eps_values=eps_values_test,
    cos_theta_values=cos_theta_values_test,
    Echi_grid=Echi_grid_test,
    Ep_max=2e3,
    n_Ep=6,
    n_k2=12,
    n_E0=10,
    n_cos=12
)
df_test = df_test.drop(columns=["eps"], errors="ignore")

print("Columnas:", list(df_test.columns))
print("Tiene eps?", "eps" in df_test.columns)
print(df_test.head())
print(f"Tiempo validación: {time.time() - t_test:.2f} s")

Columnas: ['eps2', 'cos_theta', 'm(GeV)', 'Energy', 'flux_differential']
Tiene eps? False
   eps2  cos_theta  m(GeV)    Energy  flux_differential
0   1.0        0.0    0.05  0.100000           0.000004
1   1.0        1.0    0.05  0.100000           0.000004
2   1.0        0.0    0.05  0.158489           0.000006
3   1.0        1.0    0.05  0.158489           0.000006
4   1.0        0.0    0.05  0.251189           0.000006
Tiempo validación: 0.03 s


In [10]:
# Generar UN solo CSV con varios eps2 en la misma columna
from pathlib import Path

base_csv = Path("PB_differential_flux_table.csv")
eps2_values = [5.0e-09, 8.0e-09, 9.5e-09, 1.0e-08, 2.0e-08, 5.0e-08, 8.0e-08,
       1.0e-07, 2.0e-07, 5.0e-07, 8.0e-07, 1.0e-06, 2.0e-06, 5.0e-06,
       8.0e-06, 1.0e-05, 2.0e-05, 5.0e-05, 8.0e-05, 1.0e-04, 2.0e-04,
       5.0e-04, 8.0e-04, 1.0e-03, 2.0e-03, 5.0e-03, 8.0e-03, 1.0e-02,
       5.0e-02, 1.0e-01]  
output_csv = Path("PB_differential_flux_table_all_eps2.csv")

if not base_csv.exists():
    raise FileNotFoundError(f"No existe el archivo base: {base_csv}")

df_base = pd.read_csv(base_csv)
if "flux_differential" not in df_base.columns:
    raise KeyError("No encuentro la columna 'flux_differential' en el CSV base")

# Seguridad por si existe en versiones viejas
df_base = df_base.drop(columns=["eps"], errors="ignore")

dfs = []
for eps2 in eps2_values:
    df_tmp = df_base.copy()
    df_tmp["eps2"] = float(eps2)
    df_tmp["flux_differential"] = df_base["flux_differential"].astype(float) * float(eps2)
    dfs.append(df_tmp)

df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv(output_csv, index=False)

print(f"Archivo combinado guardado: {output_csv}")
print(f"Filas totales: {len(df_all)}")
print("eps2 únicos:", sorted(df_all["eps2"].unique()))

Archivo combinado guardado: PB_differential_flux_table_all_eps2.csv
Filas totales: 114000
eps2 únicos: [np.float64(5e-09), np.float64(8e-09), np.float64(9.5e-09), np.float64(1e-08), np.float64(2e-08), np.float64(5e-08), np.float64(8e-08), np.float64(1e-07), np.float64(2e-07), np.float64(5e-07), np.float64(8e-07), np.float64(1e-06), np.float64(2e-06), np.float64(5e-06), np.float64(8e-06), np.float64(1e-05), np.float64(2e-05), np.float64(5e-05), np.float64(8e-05), np.float64(0.0001), np.float64(0.0002), np.float64(0.0005), np.float64(0.0008), np.float64(0.001), np.float64(0.002), np.float64(0.005), np.float64(0.008), np.float64(0.01), np.float64(0.05), np.float64(0.1)]
